In [14]:
import spacy
nlp = spacy.load("en_core_web_md")

In [ ]:
import pandas as pd
import numpy as np

df = pd.read_json('news_dataset (1).json')
print(df.shape)
df.head()

(7500, 2)


,text,category
0,"Larry Nassar Blames His Victims, Says He 'Was ...",CRIME
1,"Woman Beats Cancer, Dies Falling From Horse",CRIME
2,Vegas Taxpayers Could Spend A Record $750 Mill...,SPORTS
3,This Richard Sherman Interception Literally Sh...,SPORTS
4,7 Things That Could Totally Kill Weed Legaliza...,BUSINESS


In [16]:
df.category.value_counts()

category
CRIME       2500
SPORTS      2500
BUSINESS    2500
Name: count, dtype: int64

In [17]:
df['label_num']=df.category.map({
    'CRIME':0,
    'SPORTS':1,
    'BUSINESS':2
})
df.head()

,text,category,label_num
0,"Larry Nassar Blames His Victims, Says He 'Was ...",CRIME,0
1,"Woman Beats Cancer, Dies Falling From Horse",CRIME,0
2,Vegas Taxpayers Could Spend A Record $750 Mill...,SPORTS,1
3,This Richard Sherman Interception Literally Sh...,SPORTS,1
4,7 Things That Could Totally Kill Weed Legaliza...,BUSINESS,2


In [21]:
nlp = spacy.load("en_core_web_md")
def preprocess(text):
    doc = nlp(text)
    filtered_token = []
    for token in doc:
        if token.is_stop and token.is_punct:
            continue
        filtered_token.append(token.lemma_)
    return " ".join(filtered_token)

In [22]:
df['preprocess_text']=df.text.apply(lambda x:preprocess(x))

In [23]:
df.head()

,text,category,label_num,preprocess_text
0,"Larry Nassar Blames His Victims, Says He 'Was ...",CRIME,0,"Larry Nassar blame his victim , say he ' be vi..."
1,"Woman Beats Cancer, Dies Falling From Horse",CRIME,0,"Woman Beats Cancer , dies fall from horse"
2,Vegas Taxpayers Could Spend A Record $750 Mill...,SPORTS,1,Vegas Taxpayers could spend a Record $ 750 mil...
3,This Richard Sherman Interception Literally Sh...,SPORTS,1,this Richard Sherman Interception literally sh...
4,7 Things That Could Totally Kill Weed Legaliza...,BUSINESS,2,7 thing that could totally kill Weed Legalizat...


In [24]:
df['vector'] = df.preprocess_text.apply(lambda x:nlp(x).vector)

In [25]:
df.head()

,text,category,label_num,preprocess_text,vector
0,"Larry Nassar Blames His Victims, Says He 'Was ...",CRIME,0,"Larry Nassar blame his victim , say he ' be vi...","[-0.72655565, 0.14143084, -0.23835263, -0.0749..."
1,"Woman Beats Cancer, Dies Falling From Horse",CRIME,0,"Woman Beats Cancer , dies fall from horse","[-0.689915, 0.23926938, -0.0969645, -0.0507759..."
2,Vegas Taxpayers Could Spend A Record $750 Mill...,SPORTS,1,Vegas Taxpayers could spend a Record $ 750 mil...,"[-0.69217616, 0.12067247, -0.08002179, -0.0813..."
3,This Richard Sherman Interception Literally Sh...,SPORTS,1,this Richard Sherman Interception literally sh...,"[-0.6795667, 0.27941385, -0.015576292, -0.1828..."
4,7 Things That Could Totally Kill Weed Legaliza...,BUSINESS,2,7 thing that could totally kill Weed Legalizat...,"[-0.746401, 0.25002497, -0.15512888, -0.174583..."


In [27]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(
    df.vector.values,
    df.label_num,
    test_size = 0.2,
    random_state=42,
    stratify=df.label_num
)

In [29]:
import numpy as np
X_train_2d = np.stack(X_train)
X_test_2d = np.stack(X_test)

In [30]:
X_train_2d.shape

(6000, 300)

In [31]:
from sklearn.naive_bayes import MultinomialNB
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import classification_report

scaler = MinMaxScaler()
scaled_train = scaler.fit_transform(X_train_2d)
scaled_test = scaler.transform(X_test_2d)

clf = MultinomialNB()
clf.fit(scaled_train,y_train)
y_pred = clf.predict(scaled_test)
print(classification_report(y_test,y_pred))

              precision    recall  f1-score   support

           0       0.89      0.77      0.82       500
           1       0.74      0.76      0.75       500
           2       0.74      0.82      0.78       500

    accuracy                           0.78      1500
   macro avg       0.79      0.78      0.78      1500
weighted avg       0.79      0.78      0.78      1500



In [33]:
from sklearn.ensemble import GradientBoostingClassifier
clf = GradientBoostingClassifier()
clf.fit(X_train_2d,y_train)
y_pred=clf.predict(X_test_2d)
print(classification_report(y_test,y_pred))

              precision    recall  f1-score   support

           0       0.87      0.87      0.87       500
           1       0.85      0.84      0.84       500
           2       0.86      0.87      0.86       500

    accuracy                           0.86      1500
   macro avg       0.86      0.86      0.86      1500
weighted avg       0.86      0.86      0.86      1500

